In [ ]:
import json
import os
import re
import ollama
import matplotlib.pyplot as plt
import matplotlib.image as mpimg


# CONFIGURACIÓN


RUTA_JSON_TEST = "./EXIST 2026 Dataset V0.1/EXIST 2026 Memes Dataset/training/EXIST2026_training.json"

CARPETA_BASE_IMAGENES = "./EXIST 2026 Dataset V0.1/EXIST 2026 Memes Dataset/training"

RUTA_SALIDA = "predicciones_exist2026.json"

MODELO_VISION = "llama3.2-vision"

MODELO_TEXTO="qwen2.5"


# PROMPTS


PROMPT_AGENTE_A = """
Eres una mujer antifeminista moderada.

Evalúa si el meme es sexista hacia mujeres, etiquetando entre YES o NO. 
Tu postura debe resumirse así: humor negro NO, sarcasmo NO, insultos genéricos NO, body shaming ambiguo NO, 
estereotipos leves NO, bromas de internet NO, salvo que exista desprecio explícito hacia las mujeres, discriminación YES, 
degradación YES, cosificación explícita YES


Devuelve SOLO JSON válido.

NO escribas:
- markdown
- explicaciones
- ```json
- texto extra

Formato OBLIGATORIO:

{
  "analisis": "frase corta",
  "veredicto_final": "YES"
}

o

{
  "analisis": "frase corta",
  "veredicto_final": "NO"
}
"""

PROMPT_AGENTE_B = """
Eres una mujer feminista estructural.

Evalúa si el meme es sexista hacia mujeres, etiquetando entre YES o NO.
Tu postura debe resumirse así: discriminación YES, degradación YES, cosificación YES, 
estereotipos de género YES, body shaming hacia mujeres YES, sexualización ofensiva YES, 
bromas misóginas YES, humor basado en desprecio femenino YES, sarcasmo degradante YES, 
roles tradicionales impuestos YES; pero contenido neutral NO,
humor sin referencia de género NO, insultos no dirigidos específicamente a mujeres NO y memes absurdos sin carga de género NO.

Devuelve SOLO JSON válido.

NO escribas:
- markdown
- explicaciones
- ```json
- texto extra

Formato OBLIGATORIO:

{
  "analisis": "frase corta",
  "veredicto_final": "YES"
}

o

{
  "analisis": "frase corta",
  "veredicto_final": "NO"
}
"""

PROMPT_JUEZ = """
Eres un juez neutral.

Tienes que decidir si un meme es sexista o no hacia las mujeres, basandote exclusivamente en el analisis que vas a recibir por parte de 2 expertos.


Etiquetando entre YES o NO, tu postura debe resumirse así: 
discriminación explícita YES, degradación clara YES, cosificación explícita YES, misoginia directa YES, 
estereotipos jerárquicos YES, exclusión por género YES; pero humor negro NO, sátira NO, provocación NO, crítica política NO, 
sarcasmo NO, insultos genéricos NO, memes absurdos NO, ironía ambigua NO y estereotipos leves NO, 
salvo que exista una carga discriminatoria clara y verificable hacia las mujeres.

Devuelve SOLO JSON válido.

NO escribas:
- markdown
- explicaciones
- ```json
- texto extra

Formato OBLIGATORIO:

{
  "analisis": "frase corta",
  "veredicto_final": "YES"
}

o

{
  "analisis": "frase corta",
  "veredicto_final": "NO"
}
"""


# FUNCIONES AUXILIARES


def limpiar_texto(texto, max_chars=400):

    texto = texto.replace("\u00ad", "")
    texto = texto.replace("­­­­­­­­", "")

    texto = re.sub(r'(.)\1{10,}', r'\1', texto)

    texto = texto[:max_chars]

    return texto.strip()


def extraer_json_seguro(respuesta):

    try:

        match = re.search(r'\{.*\}', respuesta, re.DOTALL)

        if match:
            return json.loads(match.group())

    except Exception:
        pass

    return None


def calcular_mayoria(votos):

    votos_yes = votos.count("YES")
    votos_no = votos.count("NO")

    if votos_yes > votos_no:
        return "YES", votos_yes, votos_no

    elif votos_no > votos_yes:
        return "NO", votos_yes, votos_no

    else:
        return "EMPATE", votos_yes, votos_no


def mostrar_imagen(ruta_img):

    if os.path.exists(ruta_img):

        img = mpimg.imread(ruta_img)

        plt.figure(figsize=(4, 4))
        plt.imshow(img)
        plt.axis("off")
        plt.show()



# LLAMADAS MODELOS


def llamar_modelo_vision(prompt, texto_meme, ruta_imagen):

    mensaje = f"""
TEXTO DEL MEME:
{texto_meme}

IMPORTANTE:
- El meme está en español coloquial
- No traduzcas al inglés
- No inventes contexto raro

{prompt}
"""

    try:

        response = ollama.chat(
            model=MODELO_VISION,
            messages=[
                {
                    "role": "user",
                    "content": mensaje,
                    "images": [ruta_imagen]
                }
            ],
            options={
                "temperature": 0.2,
                "top_p": 0.9,
                "repeat_penalty": 1.2,
                "num_predict": 140
            }
        )

        salida = response["message"]["content"]

        return limpiar_texto(salida)

    except Exception as e:

        print(f"ERROR MODELO VISION: {e}")

        return json.dumps({
            "analisis": "error",
            "veredicto": "NO"
        })


def llamar_modelo_texto(prompt, contenido):

    try:

        response = ollama.chat(
            model=MODELO_TEXTO,
            messages=[
                {
                    "role": "user",
                    "content": contenido + "\n\n" + prompt
                }
            ],
            options={
                "temperature": 0.4,
                "top_p": 0.9,
                "repeat_penalty": 1.2,
                "num_predict": 140
            }
        )

        salida = response["message"]["content"]

        return limpiar_texto(salida)

    except Exception as e:

        print(f"ERROR JUEZ: {e}")

        return json.dumps({
            "analisis": "error",
            "veredicto_final": "NO"
        })



# CARGAR DATASET


with open(RUTA_JSON_TEST, "r", encoding="utf-8") as f:

    dataset_test = json.load(f)

# NUMERO DE MEMES A EVALUAR
memes_prueba = list(dataset_test.items())[:250]

print(f"\nDataset cargado. Evaluando {len(memes_prueba)} memes...\n")

predicciones = []

aciertos = 0


for id_meme, info_meme in memes_prueba:

    id_exist = info_meme["id_EXIST"]

    texto_meme = info_meme.get("text", "")

    ruta_imagen = os.path.join(
        CARPETA_BASE_IMAGENES,
        info_meme["path_memes"]
    )

    # Extracción de votos humanos

    votos_humanos = info_meme.get(
        "labels_task2_1",
        []
    )

    mayoria_real, votos_yes, votos_no = calcular_mayoria(
        votos_humanos
    )

    print("\n" + "=" * 80)
    print(f"MEME ID: {id_exist}")
    print(f"TEXTO: {texto_meme}")
    print(f"MAYORÍA HUMANA: {mayoria_real}")
    print(f"YES: {votos_yes} | NO: {votos_no}")
    print("=" * 80)

    if not os.path.exists(ruta_imagen):

        print("Imagen no encontrada")

        continue

    mostrar_imagen(ruta_imagen)


    # AGENTE A


    respuesta_A = llamar_modelo_vision(
        PROMPT_AGENTE_A,
        texto_meme,
        ruta_imagen
    )

    print("\n--- AGENTE A ---")
    print(respuesta_A)

    json_A = extraer_json_seguro(respuesta_A)

    if json_A is None:

        json_A = {
            "analisis": "respuesta inválida",
            "veredicto": "NO"
        }


    # AGENTE B


    respuesta_B = llamar_modelo_vision(
        PROMPT_AGENTE_B,
        texto_meme,
        ruta_imagen
    )

    print("\n--- AGENTE B ---")
    print(respuesta_B)

    json_B = extraer_json_seguro(respuesta_B)

    if json_B is None:

        json_B = {
            "analisis": "respuesta inválida",
            "veredicto": "NO"
        }

    # JUEZ

    contexto_juez = f"""

AGENTE A:
Analisis: {json_A.get("analisis", "")}
Veredicto: {json_A.get("veredicto", "")}

AGENTE B:
Analisis: {json_B.get("analisis", "")}
Veredicto: {json_B.get("veredicto", "")}
"""

    respuesta_juez = llamar_modelo_texto(
        PROMPT_JUEZ,
        contexto_juez
    )

    print("\n--- JUEZ ---")
    print(respuesta_juez)

    json_juez = extraer_json_seguro(
        respuesta_juez
    )

    if json_juez is None:

        json_juez = {
            "analisis": "respuesta inválida",
            "veredicto_final": "NO"
        }

    veredicto_final = json_juez.get(
        "veredicto_final",
        "NO"
    )


    # COMPARACIÓN FINAL


    if mayoria_real == "EMPATE":

        resultado = "EMPATE HUMANO"

    elif veredicto_final == mayoria_real:

        resultado = "ACIERTO"

        aciertos += 1

    else:

        resultado = "FALLO"

    print("\n" + "-" * 60)
    print(f"VEREDICTO MODELO: {veredicto_final}")
    print(f"MAYORÍA HUMANA : {mayoria_real}")
    print(f"RESULTADO       : {resultado}")
    print("-" * 60)

    # GUARDAR RESULTADO


    predicciones.append({

        "test_case": "EXIST2026",

        "id": id_exist,

        "texto_meme": texto_meme,

        "prediccion_modelo": veredicto_final,

        "mayoria_humana": mayoria_real,

        "votos_yes": votos_yes,

        "votos_no": votos_no,

        "resultado": resultado,

        "agente_A": json_A,

        "agente_B": json_B,

        "juez": json_juez
    })

# MÉTRICAS FINALES


total_validos = sum(
    1 for x in predicciones
    if x["mayoria_humana"] != "EMPATE"
)

if total_validos > 0:

    accuracy = aciertos / total_validos

else:

    accuracy = 0

print("\n" + "=" * 80)
print("RESULTADOS FINALES")
print("=" * 80)

print(f"Memes evaluados: {len(predicciones)}")
print(f"Aciertos reales: {aciertos}")
print(f"Accuracy: {accuracy:.2f}")

In [ ]:
# MÉTRICAS FINALES

# contadores


tp = 0
fp = 0
fn = 0
tn = 0


A_yes = 0
A_no = 0

B_yes = 0
B_no = 0

ambos_yes = 0
ambos_no = 0

juez_da_razon_A = 0
juez_da_razon_B = 0

juez_cambia_veredicto = 0


# RECORRER PREDICCIONES


for x in predicciones:

    mayoria = x["mayoria_humana"]

    # Ignorar empates humanos para métricas
    if mayoria == "EMPATE":
        continue

    pred = x["prediccion_modelo"]

    agente_A = x["agente_A"].get(
        "veredicto_final",
        x["agente_A"].get("veredicto", "NO")
    )

    agente_B = x["agente_B"].get(
        "veredicto_final",
        x["agente_B"].get("veredicto", "NO")
    )


    # NORMALIZAR


    agente_A = agente_A.upper()
    agente_B = agente_B.upper()
    pred = pred.upper()
    mayoria = mayoria.upper()


    # CONTADORES AGENTE A


    if agente_A == "YES":
        A_yes += 1
    else:
        A_no += 1


    # CONTADORES AGENTE B


    if agente_B == "YES":
        B_yes += 1
    else:
        B_no += 1


    # AMBOS IGUALES


    if agente_A == "YES" and agente_B == "YES":

        ambos_yes += 1

        # juez cambia aunque ambos coincidan
        if pred != "YES":
            juez_cambia_veredicto += 1

    elif agente_A == "NO" and agente_B == "NO":

        ambos_no += 1

        if pred != "NO":
            juez_cambia_veredicto += 1


    # AGENTES DISTINTOS


    elif agente_A != agente_B:

        if pred == agente_A:
            juez_da_razon_A += 1

        elif pred == agente_B:
            juez_da_razon_B += 1


    # MATRIZ CONFUSIÓN


    if pred == "YES" and mayoria == "YES":
        tp += 1

    elif pred == "YES" and mayoria == "NO":
        fp += 1

    elif pred == "NO" and mayoria == "YES":
        fn += 1

    elif pred == "NO" and mayoria == "NO":
        tn += 1


# CALCULAR F1 SCORE


precision = 0
recall = 0
f1 = 0

if (tp + fp) > 0:
    precision = tp / (tp + fp)

if (tp + fn) > 0:
    recall = tp / (tp + fn)

if (precision + recall) > 0:
    f1 = 2 * (precision * recall) / (precision + recall)


# MOSTRAR RESULTADOS


print("\n" + "=" * 80)
print("RESULTADOS FINALES")
print("=" * 80)

print(f"Memes evaluados: {len(predicciones)}")

print("\n" + "=" * 50)
print("MATRIZ DE CONFUSIÓN")
print("=" * 50)

print(f"""
                    REAL
                YES         NO
PRED YES      {tp:<10}  {fp:<10}
PRED NO       {fn:<10}  {tn:<10}
""")

print("\n--- MÉTRICAS ---")
print(f"\nPrecision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1 Score:  {f1:.4f}")


# TABLA RESUMEN AGENTES


print("\n" + "=" * 80)
print("RESUMEN MULTI-AGENTE")
print("=" * 80)

print(f"\nAGENTE A -> YES: {A_yes}")
print(f"AGENTE A -> NO : {A_no}")

print(f"\nAGENTE B -> YES: {B_yes}")
print(f"AGENTE B -> NO : {B_no}")

print(f"\nAMBOS YES: {ambos_yes}")
print(f"AMBOS NO : {ambos_no}")

print(f"\nJUEZ DA RAZÓN A A: {juez_da_razon_A}")
print(f"JUEZ DA RAZÓN A B: {juez_da_razon_B}")

print(f"\nJUEZ CAMBIA CONSENSO: {juez_cambia_veredicto}")